# TicketRoute: Transformer training and evaluation

Upload the original project ZIP. The bundled synthetic dataset is only a smoke test, not a benchmark. You may select a GPU runtime, but availability is not guaranteed. CPU training works and can be slow.

In [ ]:
from google.colab import files
uploaded = files.upload()


In [ ]:
from pathlib import Path
import zipfile
archives = [Path(name) for name in uploaded if name.endswith(".zip")]
if len(archives) != 1:
    raise ValueError("Upload exactly one TicketRoute ZIP.")
destination = Path("/content/ticketroute_workspace")
destination.mkdir(exist_ok=True)
with zipfile.ZipFile(archives[0]) as archive:
    for item in archive.infolist():
        target = (destination / item.filename).resolve()
        if not target.is_relative_to(destination.resolve()):
            raise ValueError("Unsafe archive path")
    archive.extractall(destination)
project = destination / "ticketroute"
assert (project / "train.py").exists()
%cd $project


## Install dependencies
If a restart is requested, restart and rerun folder setup. If the runtime has a compatible CUDA-enabled Torch version, retain it.

In [ ]:
%pip install -r requirements-train.txt


In [ ]:
import torch
print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("Training will use CPU and can take longer.")


## Data checks
Do not run create_sample_data.py over your custom data: it overwrites CSVs.

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "data_utils.py"], check=True)
subprocess.run([sys.executable, "-m", "unittest", "discover", "-s", "tests", "-v"], check=True)


## Train and evaluate
This downloads the base model, trains the new classification head and encoder, and saves `models/fine_tuned/`. The eight test examples make metrics unstable. The evaluation also downloads the separate zero-shot model.

In [ ]:
subprocess.run([sys.executable, "train.py"], check=True)


In [ ]:
subprocess.run([sys.executable, "evaluate.py", "--mode", "both", "--split", "test", "--include-oos"], check=True)


## Save your trained model and reports before ending the runtime
Extract the output ZIP inside your local ticketroute folder. For a proper experiment, choose thresholds using validation before a single final evaluation of a larger held-out set.

In [ ]:
import json, platform, zipfile
manifest = {"python": platform.python_version(), "torch": torch.__version__, "cuda": torch.cuda.is_available()}
if torch.cuda.is_available():
    manifest["gpu"] = torch.cuda.get_device_name(0)
Path("run_manifest.json").write_text(json.dumps(manifest, indent=2))
freeze = subprocess.check_output([sys.executable, "-m", "pip", "freeze"], text=True)
Path("training_environment.txt").write_text(freeze)
output = Path("/content/TicketRoute_Trained_Artifacts.zip")
with zipfile.ZipFile(output, "w", zipfile.ZIP_DEFLATED) as archive:
    for folder in [Path("models/fine_tuned"), Path("reports")]:
        for path in folder.rglob("*"):
            if path.is_file(): archive.write(path, path.as_posix())
    for name in ["run_manifest.json", "training_environment.txt"]:
        archive.write(name, name)
files.download(str(output))
